**Yayika Tinner - 6722781026**

AI usage: Claude for explanations, debugging help, reword my explanations, checking my code & answers, and made the grader notes

## Part 1 - The honest build

#### 1. Setup and fingerprint

In [1]:
import sys
import numpy as np
import sklearn
from sklearn.dummy import DummyClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import TimeSeriesSplit, cross_val_score

# version fingerprint
print("python", sys.version.split()[0], "| numpy", np.__version__, "| sklearn", sklearn.__version__, "| seeds: default_rng(322), random_state=0")

python 3.12.10 | numpy 2.5.3 | sklearn 1.9.1 | seeds: default_rng(322), random_state=0


#### 2. Make the data

In [2]:
rng = np.random.default_rng(322)          # fixed seed
t = np.arange(730)                        # 730 days, t = [0, 1, ..., 729]

# smooth yearly wave: centre 25, swings ~18, repeat every 365 days, peak at day 15
seasonal = 25 + 18 * np.cos(2 * np.pi * (t - 15) / 365)

# add noise: noise today = 0.9 * noise yesterday + random num centered on 0 and swing ~7
noise = np.zeros(730)
for i in range(1, 730):
    noise[i] = 0.9 * noise[i - 1] + rng.normal(0, 7.0)
    
pm = np.clip(seasonal + noise, 2, None)   # PM2.5 can't go below 2

LAGS = 3                                  # last 3 days
# make 3 rows with the pm values, then change them to be columns, so each row has 3 pm values
X = np.column_stack([pm[LAGS - 1 - k : -1 - k] for k in range(LAGS)])
y = (pm[LAGS:] > 37.5).astype(int)        # label, 1 = tmr dangerous, 0 = safe

print("X shape:", X.shape, " y shape:", y.shape)
print("base rate:", y.mean().round(3))    # base rate: dangerous days

X shape: (727, 3)  y shape: (727,)
base rate: 0.27


The base rate is 0.27, so about 27% of days are dangerous (tomorrow above 37.5) and 73% are safe. Most days are safe, so a model can get high accuracy just by saying "safe". Any model has to beat that.

#### 3. Time split and the rock

In [3]:
n_tr = int(0.75 * len(y))                 # first 75% = train

# X = pm of last 3 days, y = 1/0 for next day
X_train, y_train = X[:n_tr], y[:n_tr]
X_test,  y_test  = X[n_tr:], y[n_tr:]
print("train rows:", len(y_train), " test rows:", len(y_test))

rock = DummyClassifier(strategy="most_frequent").fit(X_train, y_train)
print("rock accuracy on test:", round(rock.score(X_test, y_test), 3))
print("dangerous test days:", y_test.sum(), "of", len(y_test))

train rows: 545  test rows: 182
rock accuracy on test: 0.742
dangerous test days: 47 of 182


Split by time: the first 75% of days (545) are for training and the last 25% (182) are for testing. In real scenario the model only has past days. A shuffled split would let it see days right next to the test days, which makes the score better than it really is.

The rock always says "safe" and gets 0.742. That number is the share of safe days: 47 of 182 days are dangerous (0.258), so 1 - 0.258 = 0.742.

#### 4. The honest model, inside a pipeline

In [4]:
pipe = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
pipe.fit(X_train, y_train)                # scaler and model learn from train only
print("model accuracy on test:", round(pipe.score(X_test, y_test), 3))

model accuracy on test: 0.874


The model gets 0.874 on the test set, about 0.13 better than the rock (0.742). The scaler and the model are inside one pipeline, so the scaler learns its mean and spread from the training days only and never sees the test days.

#### 5. TimeSeriesSplit, model against rock, per fold

In [5]:
cv = TimeSeriesSplit(n_splits=5)
model_scores = cross_val_score(pipe, X, y, cv=cv)
rock_scores = cross_val_score(DummyClassifier(strategy="most_frequent"), X, y, cv=cv)

print("fold | test days | dangerous | rock | model")
for i, (tr, te) in enumerate(cv.split(X), start=1):
    print(f"  {i}  |   {len(te):3d}     |    {y[te].sum():3d}    | {rock_scores[i-1]:.2f} | {model_scores[i-1]:.2f}")

print("model mean", model_scores.mean().round(3), " std", model_scores.std().round(3))
print("rock  mean", rock_scores.mean().round(3), " std", rock_scores.std().round(3))

fold | test days | dangerous | rock | model
  1  |   121     |      0    | 1.00 | 1.00
  2  |   121     |     37    | 0.69 | 0.93
  3  |   121     |     59    | 0.51 | 0.78
  4  |   121     |      7    | 0.94 | 0.96
  5  |   121     |     40    | 0.67 | 0.86
model mean 0.904  std 0.079
rock  mean 0.764  std 0.181


Across 5 time-ordered folds, the model scores 0.904 ± 0.079 and the rock 0.764 ± 0.181. The model beats or ties the rock in every fold. The rock's spread is bigger because its score is just the share of safe days, which changes a lot between seasons.

Fold 1 scores 1.00 for both, but this is a warning, not a success. Its 121 test days contain 0 dangerous days, so always saying "safe" is never wrong, and the fold cannot show whether the model catches dangerous days.

Fold 4 has only 7 dangerous days, so the rock already gets 0.94 and the model only 0.96 (about 2 more correct days). When dangerous days are rare, accuracy looks high for both the rock and model and hides whether the dangerous days are caught.

The last two cross-validation folds overlap the test period. I only report CV, and no choice depends on it.

## Part 2 - Break and repair

#### 1. Wrong score 1 - Shuffled split

In [6]:
from sklearn.model_selection import train_test_split

# WRONG: shuffle the days before splitting
Xs_tr, Xs_te, ys_tr, ys_te = train_test_split(X, y, test_size=0.25, shuffle=True, random_state=0)

shuf = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))   # same model as part 1
shuf.fit(Xs_tr, ys_tr)

rock_s = DummyClassifier(strategy="most_frequent").fit(Xs_tr, ys_tr)        # rock on the same shuffled days
print("rock on shuffled test days:", round(rock_s.score(Xs_te, ys_te), 3))

print("WRONG shuffled accuracy:", round(shuf.score(Xs_te, ys_te), 3))
print("Honest time-split accuracy:", round(pipe.score(X_test, y_test), 3))

rock on shuffled test days: 0.703
WRONG shuffled accuracy: 0.918
Honest time-split accuracy: 0.874


The shuffled split gives 0.918, higher than the honest 0.874, with the rock = 0.703. Shuffling mixes the days before splitting, so many test days end up between training days. Neighbouring days are very similar (the noise is 0.9 times yesterday's), and a test day's reading even shows up as "yesterday" in the next day's training row. So the model fills in gaps instead of forecasting the future, and the score goes up by 0.044.

#### 2. Wrong score 2 - Centred smoothed column

In [7]:
# 3-day average centred on today = (yesterday + today + tomorrow) / 3
centred = np.convolve(pm, np.ones(3) / 3, mode="same")
X_leak = np.column_stack([X, centred[LAGS - 1 : -1]])    # add centered avg as the 4th column

leak = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
leak.fit(X_leak[:n_tr], y[:n_tr])                         # same time split as part 1

print("WRONG centred-column accuracy:", round(leak.score(X_leak[n_tr:], y[n_tr:]), 3))
print("Honest time-split accuracy:", round(pipe.score(X_test, y_test), 3))
print()

# proof for the first row: the new column uses tomorrow's reading (pm[3])
print("first row:", X_leak[0].round(1), " tomorrow's pm:", pm[3].round(1))
print("(yesterday + today + tomorrow) / 3 =", ((pm[1] + pm[2] + pm[3]) / 3).round(1))

WRONG centred-column accuracy: 0.929
Honest time-split accuracy: 0.874

first row: [54.7 46.5 42.4 51.1]  tomorrow's pm: 52.2
(yesterday + today + tomorrow) / 3 = 51.1


Adding a 3-day average centred on today raises the score to 0.929, even though the time split is the same as in Part 1. The average is (yesterday + today + tomorrow) / 3, so tomorrow's reading, the thing we want to predict, is hidden inside the column. The printout proves it: the first row's new column is 51.1, which is the average including tomorrow's 52.2. A split only decides which rows go to train or test, it never looks inside a column, so even a correct split cannot catch this leak.

#### 3. Repair - past-only smoothed column (for wrong score 2)

In [8]:
# 3-day average of the past only = (2 days ago + yesterday + today) / 3
trailing = np.convolve(pm, np.ones(3) / 3, mode="valid")[:-1]   # not take last avg
X_fix = np.column_stack([X, trailing])                    # same 4th column, but past only

fix = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
fix.fit(X_fix[:n_tr], y[:n_tr])                           # same time split again
print("REPAIRED trailing-column accuracy:", round(fix.score(X_fix[n_tr:], y[n_tr:]), 3))
print("Honest time-split accuracy:", round(pipe.score(X_test, y_test), 3))
print()

# proof for the first row: the new column only uses days already known
print("first row:", X_fix[0].round(1))
print("(2 days ago + yesterday + today) / 3 =", ((pm[0] + pm[1] + pm[2]) / 3).round(1))
print()

print("trailing = mean of my 3 lag columns:", np.allclose(trailing, X.mean(axis=1)))

REPAIRED trailing-column accuracy: 0.874
Honest time-split accuracy: 0.874

first row: [54.7 46.5 42.4 47.9]
(2 days ago + yesterday + today) / 3 = 47.9

trailing = mean of my 3 lag columns: True


To repair it, I used a trailing window: (2 days ago + yesterday + today) / 3, which we already know. The score drops back to 0.874 (= the honest score from Part 1), so the +0.055 accuracy came from tomorrow leaking data. The trailing average is just the mean of my three lag columns (checked above), so it adds no new information and getting exactly 0.874 back is expected.

## Part 3 - The cut-off policy

#### 1. Costs and choosing the cut-off on a validation set

In [9]:
from sklearn.metrics import confusion_matrix

COST_MISS = 10      # dangerous day called safe: children breathe bad air
COST_ALARM = 1      # safe day called dangerous: unneeded indoor day

# split TRAINING days by time
n_fit = int(0.75 * n_tr)
X_fit, y_fit = X_train[:n_fit], y_train[:n_fit] # first 75% of training -> fit
X_val, y_val = X_train[n_fit:], y_train[n_fit:] # last 25% of training -> pick cut-off
print("fit days:", len(y_fit), " validation days:", len(y_val), " dangerous in validation:", y_val.sum())

val_pipe = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
val_pipe.fit(X_fit, y_fit)
p_val = val_pipe.predict_proba(X_val)[:, 1]          # probability of danger for each validation day

print("cut-off | false alarms | misses | cost")
best_cut, best_cost = None, None
for cut in np.round(np.arange(0.05, 1, 0.05), 2):
    tn, fp, fn, tp = confusion_matrix(y_val, (p_val >= cut).astype(int), labels=[0, 1]).ravel()
    cost = COST_MISS * fn + COST_ALARM * fp
    print(f"  {cut:.2f}  |      {fp:3d}     |  {fn:3d}   | {cost:4d}")
    if best_cost is None or cost < best_cost:
        best_cut, best_cost = cut, cost

print("chosen cut-off (lowest validation cost):", best_cut, " cost:", best_cost)

fit days: 408  validation days: 137  dangerous in validation: 30
cut-off | false alarms | misses | cost
  0.05  |       38     |    0   |   38
  0.10  |       24     |    0   |   24
  0.15  |       20     |    0   |   20
  0.20  |       16     |    1   |   26
  0.25  |       12     |    2   |   32
  0.30  |        9     |    3   |   39
  0.35  |        9     |    3   |   39
  0.40  |        8     |    4   |   48
  0.45  |        7     |    7   |   77
  0.50  |        5     |    7   |   75
  0.55  |        5     |    9   |   95
  0.60  |        4     |   11   |  114
  0.65  |        3     |   12   |  123
  0.70  |        3     |   13   |  133
  0.75  |        2     |   15   |  152
  0.80  |        2     |   16   |  162
  0.85  |        1     |   16   |  161
  0.90  |        1     |   20   |  201
  0.95  |        0     |   26   |  260
chosen cut-off (lowest validation cost): 0.15  cost: 20


I put a cost on each mistake: a missed dangerous day costs 10 and a false alarm costs 1. A miss means children play outside in dangerous air, while a false alarm only means one unneeded indoor day, so a miss is much worse.

To choose the cut-off without touching the test set, I split the 545 training days again by time: the first 408 days train a model and the last 137 days (30 dangerous) are the validation set. I tried cut-offs from 0.05 to 0.95 and counted the cost of each one. The default 0.5 costs 75 on validation (5 false alarms, 7 misses), while 0.15 costs only 20 (20 false alarms, 0 misses), so I chose 0.15. Going lower than 0.15 does not help, because it only adds more false alarms.

10 : 1 is a policy choice. From the table above, 0.15 stays the best cut-off for any miss cost above 4. The validation set has only 30 dangerous days, so the exact cut-off is uncertain and should be reviewed with more data.

#### 2. The test set: default 0.5 against the chosen cut-off

In [10]:
p_test = pipe.predict_proba(X_test)[:, 1]            # honest model from part 1, trained on all 545 days

print("test base rate:", y_test.mean().round(3), " dangerous days:", y_test.sum(), "of", len(y_test))
for cut in [0.5, best_cut]:
    tn, fp, fn, tp = confusion_matrix(y_test, (p_test >= cut).astype(int), labels=[0, 1]).ravel()
    cost = COST_MISS * fn + COST_ALARM * fp
    print(f"cut-off {cut}: [[TN {tn}, FP {fp}], [FN {fn}, TP {tp}]]")
    print(f"   accuracy {(tn + tp) / len(y_test):.3f}  recall {tp / (tp + fn):.2f}  precision {tp / (tp + fp):.2f}  cost {cost}")

test base rate: 0.258  dangerous days: 47 of 182
cut-off 0.5: [[TN 126, FP 9], [FN 14, TP 33]]
   accuracy 0.874  recall 0.70  precision 0.79  cost 149
cut-off 0.15: [[TN 107, FP 28], [FN 3, TP 44]]
   accuracy 0.830  recall 0.94  precision 0.61  cost 58


The test base rate is 0.258 (47 dangerous days out of 182). The confusion matrix explanation:

- At the default cut-off 0.5: 126 safe days were correctly left alone, 9 safe days got a false alarm, 14 dangerous days were missed, and 33 dangerous days were caught.
- At my cut-off 0.15: 107 safe days were left alone, 28 safe days got a false alarm, only 3 dangerous days were missed, and 44 dangerous days were caught.

The new cut-off gains 11 fewer missed dangerous days and loses 19 more indoor days. Total cost falls from 149 to 58. Accuracy drops from 0.874 to 0.830 and precision from 0.79 to 0.61, but recall rises from 0.70 to 0.94. Accuracy counts a miss and a false alarm as equally bad, but for a school they are not, so lower accuracy here is the better policy.

0.5 is not a policy, it is just the library's default. Nobody chose it by thinking about what each mistake costs the school, so the cut-off should come from the costs instead.

The cut-off was chosen with a model trained on the first 408 days, then applied to the final model trained on all 545 training days. The extra 137 days can shift its probabilities a little, so the cut-off should be rechecked on new data.

## Part 4 - The AUC receipts

#### 1. AUC computed two independent ways

In [11]:
from sklearn.metrics import roc_curve, roc_auc_score

# way 1: build the ROC curve, then trapezoid rule for the area under it
fpr, tpr, cuts = roc_curve(y_test, p_test)
auc_trap = np.trapezoid(tpr, fpr)

# way 2: count every (dangerous, safe) pair, tie = half
p_danger = p_test[y_test == 1]           # 47 dangerous days
p_safe = p_test[y_test == 0]             # 135 safe days
higher = (p_danger[:, None] > p_safe[None, :]).sum()    # compare col with row
ties = (p_danger[:, None] == p_safe[None, :]).sum()
pairs = len(p_danger) * len(p_safe)
auc_pairs = (higher + 0.5 * ties) / pairs

# way 3: library, lecture p.79
auc_lib = roc_auc_score(y_test, p_test)

print("way 1 trapezoid rule :", round(auc_trap, 4))
print("way 2 pair counting  :", round(auc_pairs, 4), " (", higher, "of", pairs, "pairs, ties:", ties, ")")
print("way 3 roc_auc_score  :", round(auc_lib, 4))
print("all agree:", np.isclose(auc_trap, auc_pairs) and np.isclose(auc_pairs, auc_lib))

way 1 trapezoid rule : 0.9349
way 2 pair counting  : 0.9349  ( 5932 of 6345 pairs, ties: 0 )
way 3 roc_auc_score  : 0.9349
all agree: True


I computed the AUC 2 ways & a library check from the lecture:

Way 1 builds the ROC curve (one point per cut-off, false-alarm rate along the bottom, catch rate on the side) and finds the area under it with the trapezoid formula.

Way 2 doesn't draw a curve, it compares every pair of one dangerous day to one safe day (47 x 135 = 6,345 pairs) and counts how often the dangerous day got the higher score, with a tie counting as half. 5,932 of the 6,345 pairs are in the right order.

All 3 gives 0.9349

#### 2. What the AUC means

An AUC of 0.9349 means that if I pick one dangerous day and one safe day at random, the model gives the dangerous day the higher score 93.49% of the time.

## Memo

**Recommendation**: 
Deploy a simple model: logistic regression (a standard yes/no prediction method) that reads the last three days of Bangkok's average PM2.5 each evening and gives a probability that tomorrow will be above the 37.5 limit. The scaling step and the model sit together in one pipeline, so they only ever learn from past days.

**The number to show schools**: 
On the 182 most recent days, which the model never saw while learning, it warned schools on 44 of the 47 dangerous days (94%). The baseline of always saying "safe" warns on 0 of 47 (0%). Only 26% of the test days are dangerous, so "always safe" still looks 74% accurate while protecting no one. That is why I report dangerous days caught (recall), not accuracy.

**The alarm level and why**: 
The model gives a probability, and we must choose the level that triggers an alarm, I set it at 0.15. I counted a missed dangerous day as 10 times worse than a false alarm, because a miss means children breathe dangerous air, while a false alarm only costs 1 indoor day. I picked 0.15 on a separate set of older days, so the final test stayed untouched.

**What it trades**: 
Compared with 0.5, the new level misses 11 fewer dangerous days (3 instead of 14) but adds 19 more false alarms (28 instead of 9). Overall accuracy falls from 87% to 83%, which is expected: accuracy treats both mistakes as equal, but for schools they are not.

**Why not 0.5**: 
It is only the software's default. Nobody chose it by weighing the cost of each mistake, so it is not a policy.

**How I checked the number**: 
I tested only on later days, never shuffled ones, and removed any input that secretly used tomorrow's reading. Those mistakes made the score look better (up to 93%), so the honest numbers above are the ones to trust.

**Limits**: 
The data is generated, not real, and the test has only 47 dangerous days from one period. The model still missed 3 of 47, and 28 in 72 alarms were false. Review the alarm level after the first season.